In [0]:
from pyspark.sql import functions as f

In [0]:
start_date = "2024-01-01"
end_date   = "2025-12-01"

In [0]:
df = (
    spark.sql(f"""
              SELECT explode(
                  sequence(
                      to_date('{start_date}'), 
                      to_date('{end_date}'), 
                      interval 1 month)
              ) AS month_start_date
              """)
)

df = (
    df
    .withColumn("date_key", f.date_format("month_start_date", "yyyyMM").cast("int"))
    .withColumn("year", f.year("month_start_date"))
    .withColumn("month_name", f.date_format("month_start_date", "MMMM"))
    .withColumn("month_sort_name", f.date_format("month_start_date", "MMM"))
    .withColumn("quater", f.concat(f.lit("Q"), f.quarter("month_start_date")))
    .withColumn("year_quarter", f.concat(f.col("year"), f.lit("-Q"), f.quarter("month_start_date")))
)

In [0]:
display(df)

In [0]:
df.write.mode("overwrite")\
    .format("delta")\
    .saveAsTable("fmcg.gold.dim_date")